In [0]:
%run "../01-setup/1.configure_access_to_cloud_storage"

In [0]:
%run "../01-setup/2.common_functions"

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2025-01")
v_file_date = dbutils.widgets.get("p_file_date")
raw_race_path = f"{raw_folder_path}/{v_file_date}"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

name_schema_static = StructType([
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True),
])

drivers_schema_static = StructType([
    StructField("driverId", IntegerType(), False),
    StructField("driverRef", StringType(), True),
    StructField("number", IntegerType(), True),
    StructField("code", StringType(), True),
    StructField("name", name_schema_static),
    StructField("dob", DateType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True),
])

name_schema_incremental = StructType([
    StructField("givenName", StringType(), True),
    StructField("familyName", StringType(), True),
])

drivers_schema_incremental = StructType([
    StructField("driverId", StringType(), True),
    StructField("name", name_schema_incremental),
    StructField("dateOfBirth", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("url", StringType(), True),
])

drivers_schema = drivers_schema_incremental if USE_INCREMENTAL else drivers_schema_static

if USE_INCREMENTAL:
    drivers_df = spark.read.schema(drivers_schema).json(f"{raw_race_path}/drivers.json")
else:
    drivers_df = spark.read.schema(drivers_schema).json(f"{raw_folder_path}/drivers.json")

display(drivers_df)

In [0]:
from pyspark.sql.functions import current_timestamp, concat_ws, col

In [0]:
from pyspark.sql.functions import col, concat_ws, current_timestamp, lit

if USE_INCREMENTAL:
    drivers_final_df = (
        drivers_df
        .withColumn("name", concat_ws(" ", col("name.givenName"), col("name.familyName")))
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("dateOfBirth", "dob")
        .drop("url")
        .withColumn("ingestion_date", current_timestamp())
        .withColumn("data_source", lit(v_data_source))
        .withColumn("file_date", lit(v_file_date))
    )
else:
    drivers_final_df = (
        drivers_df
        .withColumn("name", concat_ws(" ", col("name.forename"), col("name.surname")))
        .drop("url")
        .withColumnRenamed("driverId", "driver_id")
        .withColumnRenamed("driverRef", "driver_ref")
        .withColumn("ingestion_date", current_timestamp())
    )

display(drivers_final_df)

In [0]:
drivers_final_df.write.mode("overwrite").format("delta").save(f"{processed_folder_path}/drivers")

In [0]:
df = spark.read.format("delta").load(f"{processed_folder_path}/drivers")
display(df)

In [0]:
dbutils.notebook.exit("Success")